# Retrieval Project Workbook

This notebook replaces the old notebook collection with one guided workbook.

## What you will learn
- how the dataset is structured,
- why we create a shared `content` field,
- how TF-IDF and BM25 rank documents,
- how to run a fast classroom-sized experiment,
- how the project code is organized,
- what to practice next.

The emphasis is on **memory-friendly learning**: plots, diagrams, compact experiments, and active-learning prompts.


In [ ]:
from pathlib import Path
import json
import math
import sys
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation import evaluate_retrieval
from src.models import run_bm25_search, run_tfidf_search, tokenize
from src.preprocess import create_content_column
from src.utils import build_qrels_lookup

DATA_DIR = ROOT / "data"
plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.max_colwidth = 90


## 1. Project Map

The cleaned project now separates three concerns:

1. **Execution** in `src/` and `main.py`.
2. **Learning** in this notebook.
3. **Formal reporting** in `report/retrieval_project_report.pdf`.

That separation matters because it keeps the Kaggle pipeline reproducible while still giving you a place to understand every idea slowly.


In [ ]:
docs_df = pd.read_json(DATA_DIR / "docs.json")
train_queries_df = pd.read_json(DATA_DIR / "queries_train.json")
test_queries_df = pd.read_json(DATA_DIR / "queries_test.json")
qrels_raw = json.loads((DATA_DIR / "qgts_train.json").read_text())
qrels_lookup = build_qrels_lookup(qrels_raw)

summary_df = pd.DataFrame(
    [
        {"split": "documents", "rows": len(docs_df), "columns": ", ".join(docs_df.columns)},
        {"split": "train queries", "rows": len(train_queries_df), "columns": ", ".join(train_queries_df.columns)},
        {"split": "test queries", "rows": len(test_queries_df), "columns": ", ".join(test_queries_df.columns)},
        {"split": "qrels entries", "rows": len(qrels_lookup), "columns": "query_id -> relevant_doc_ids"},
    ]
)

display(summary_df)


## 2. Build the Shared Retrieval Text

The raw files contain several text-like fields. The project keeps comparisons fair by building one normalized field called `content`.

- documents use `title + text + tags`
- queries use `title + text`

That means every model reads the same semantic input, so differences in quality come from the ranking model, not from inconsistent preprocessing.


In [ ]:
processed_docs = create_content_column(docs_df, ["title", "text", "tags"])
processed_train_queries = create_content_column(train_queries_df, ["title", "text"])

example_view = pd.DataFrame(
    {
        "doc_id": processed_docs["id"].head(3),
        "title": processed_docs["title"].head(3),
        "content_preview": processed_docs["content"].str.slice(0, 140).head(3),
    }
)

display(example_view)


In [ ]:
doc_category_counts = docs_df["category"].value_counts().sort_values(ascending=False)
doc_lengths = docs_df["text"].fillna("").astype(str).str.split().str.len()
query_lengths = train_queries_df["text"].fillna("").astype(str).str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

axes[0].bar(doc_category_counts.index, doc_category_counts.values, color=["#376996", "#4c9f70", "#d17b40", "#aa5d7c", "#d8b83f"])
axes[0].set_title("Document Count by Category")
axes[0].set_ylabel("Documents")
axes[0].tick_params(axis="x", rotation=20)

axes[1].hist(doc_lengths, bins=40, alpha=0.75, color="#376996", label="documents")
axes[1].hist(query_lengths, bins=30, alpha=0.65, color="#d17b40", label="train queries")
axes[1].set_title("Length Distribution")
axes[1].set_xlabel("Number of words")
axes[1].legend()

plt.suptitle("The Corpus at a Glance", fontsize=15, y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(13.5, 3.6))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

steps = [
    ("Load JSON", "docs / queries / qrels", "#dceaf7"),
    ("Preprocess", "build shared content", "#dff4e5"),
    ("Retrieve", "TF-IDF / BM25 / hybrid", "#f8ead7"),
    ("Evaluate", "Precision, Recall, MRR, MAP", "#f7dbe6"),
    ("Submit", "CSV + template validation", "#f8f2cf"),
]

box_width = 0.16
for idx, (title, subtitle, color) in enumerate(steps):
    x0 = 0.03 + idx * 0.19
    patch = FancyBboxPatch(
        (x0, 0.32),
        box_width,
        0.36,
        boxstyle="round,pad=0.02,rounding_size=0.03",
        linewidth=1.6,
        edgecolor="#243447",
        facecolor=color,
    )
    ax.add_patch(patch)
    ax.text(x0 + box_width / 2, 0.55, title, ha="center", va="center", fontsize=12, fontweight="bold")
    ax.text(x0 + box_width / 2, 0.42, subtitle, ha="center", va="center", fontsize=10)
    if idx < len(steps) - 1:
        arrow = FancyArrowPatch(
            (x0 + box_width, 0.5),
            (x0 + 0.19, 0.5),
            arrowstyle="simple",
            mutation_scale=18,
            color="#243447",
            linewidth=1,
        )
        ax.add_patch(arrow)

ax.set_title("Pipeline Flow: one view to remember the whole project", fontsize=15, pad=12)
plt.show()


## 3. Model Intuition Without Black Boxes

Before using library code, it helps to build intuition on a toy corpus.

We will compare lexical ranking ideas on three short documents. This is not about leaderboard quality. It is about making the scoring logic visible.


In [ ]:
toy_docs = {
    "d1": "python dictionary list comprehension tutorial",
    "d2": "linux shell command find grep tutorial",
    "d3": "python list append extend insert examples",
}
query = "python list tutorial"

def tf(tokens):
    counts = Counter(tokens)
    total = max(1, len(tokens))
    return {term: value / total for term, value in counts.items()}

corpus_tokens = {doc_id: tokenize(text) for doc_id, text in toy_docs.items()}
query_tokens = tokenize(query)
query_tf = tf(query_tokens)

idf = {}
for term in sorted({term for tokens in corpus_tokens.values() for term in tokens}):
    df = sum(1 for tokens in corpus_tokens.values() if term in tokens)
    idf[term] = math.log((1 + len(corpus_tokens)) / (1 + df)) + 1

rows = []
for doc_id, tokens in corpus_tokens.items():
    doc_tf = tf(tokens)
    score = sum(query_tf.get(term, 0.0) * doc_tf.get(term, 0.0) * idf[term] ** 2 for term in query_tf)
    rows.append({"doc_id": doc_id, "tokens": " ".join(tokens), "manual_tfidf_like_score": round(score, 4)})

display(pd.DataFrame(rows).sort_values("manual_tfidf_like_score", ascending=False))


### What to notice

- `d1` and `d3` both match `python` and `list`, but only `d1` also matches `tutorial`.
- BM25 and TF-IDF are both lexical methods, but they reward term frequency and document length differently.
- The real project later adds a hybrid semantic reranker on top of strong lexical candidates.


In [ ]:
study_category = "tex"
study_docs = processed_docs.loc[processed_docs["category"] == study_category].head(30000).copy()
study_queries = processed_train_queries.loc[processed_train_queries["category"] == study_category].head(12).copy()

study_doc_ids = set(study_docs["id"].astype(str))
study_qrels = {}
for query_id in study_queries["id"].astype(str):
    relevant = [doc_id for doc_id in qrels_lookup.get(query_id, []) if doc_id in study_doc_ids]
    if relevant:
        study_qrels[query_id] = relevant

study_queries = study_queries[study_queries["id"].astype(str).isin(study_qrels.keys())].copy()

print(f"Study subset: {len(study_docs):,} docs | {len(study_queries)} queries | category={study_category}")

tfidf_results = run_tfidf_search(study_docs, study_queries, top_k=10)
bm25_results = run_bm25_search(study_docs, study_queries, top_k=10)

tfidf_metrics = evaluate_retrieval(tfidf_results, study_qrels, k=10)
bm25_metrics = evaluate_retrieval(bm25_results, study_qrels, k=10)

metrics_df = pd.DataFrame(
    [
        {"model": "TF-IDF", **tfidf_metrics},
        {"model": "BM25+", **bm25_metrics},
    ]
)

display(metrics_df.round(4))

plot_df = metrics_df.melt(id_vars="model", var_name="metric", value_name="score")
fig, ax = plt.subplots(figsize=(10, 4.2))
for idx, model_name in enumerate(plot_df["model"].unique()):
    model_slice = plot_df[plot_df["model"] == model_name]
    ax.bar(
        np.arange(len(model_slice)) + (idx - 0.5) * 0.35,
        model_slice["score"],
        width=0.35,
        label=model_name,
        color=["#376996", "#d17b40"][idx],
    )

ax.set_xticks(np.arange(len(model_slice)))
ax.set_xticklabels(model_slice["metric"], rotation=15)
ax.set_ylim(0, max(0.05, plot_df["score"].max() * 1.2))
ax.set_title("Fast Study Benchmark on a Category Subset")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
example_query_id = next(iter(study_qrels))
example_query = study_queries.loc[study_queries["id"].astype(str) == example_query_id].iloc[0]
study_doc_lookup = study_docs.assign(id_str=study_docs["id"].astype(str)).set_index("id_str")

def inspect_titles(results, query_id, top_n=5):
    ranked = next(item["relevant_docs"] for item in results if str(item["query_id"]) == str(query_id))
    rows = []
    for rank, doc_id in enumerate(ranked[:top_n], start=1):
        row = study_doc_lookup.loc[str(doc_id)]
        rows.append({"rank": rank, "doc_id": doc_id, "title": row["title"], "category": row["category"]})
    return pd.DataFrame(rows)

print("Query title:")
print(example_query["title"])
print("\nQuery text:")
print(example_query["text"])

print("\nTF-IDF top documents")
display(inspect_titles(tfidf_results, example_query_id))

print("BM25+ top documents")
display(inspect_titles(bm25_results, example_query_id))


## 4. Active Learning Prompts

Use these as short practice loops after you understand the baseline notebook cells.

### Practice A: preprocessing
- Remove tags from document `content` and rerun the subset experiment.
- Add tags to query `content` and compare the ranking changes.
- Inspect whether lowercasing helps or hurts code-like tokens.

### Practice B: ranking
- Change `top_k` from 10 to 5 and compare Precision@K versus Recall@K.
- Restrict the corpus to another category such as `android` or `unix`.
- Compare the first relevant hit position for TF-IDF and BM25 on the same query.

### Practice C: critical reading
- When does lexical matching fail even though the user intent is obvious?
- Which errors would a semantic reranker probably fix?
- Which errors are caused by the dataset rather than the ranking model?


In [ ]:
code_map = pd.DataFrame(
    [
        {"file": "src/config.py", "role": "Central runtime configuration for the pipeline."},
        {"file": "src/preprocess.py", "role": "Builds normalized text and keeps docs/queries comparable."},
        {"file": "src/models.py", "role": "Implements TF-IDF, BM25, dense retrieval, and hybrid reranking."},
        {"file": "src/evaluation.py", "role": "Computes Precision@K, Recall@K, MRR, and MAP."},
        {"file": "src/utils.py", "role": "Loads data, writes Kaggle CSV files, and validates the template."},
        {"file": "src/pipeline.py", "role": "Connects loading, preprocessing, evaluation, submission writing, and validation."},
        {"file": "main.py", "role": "Thin entrypoint that launches the configured pipeline."},
    ]
)

display(code_map)


## 5. Appendix Notes

### Why one notebook is better here
- the explanations are no longer scattered,
- the visuals and experiments now build on each other,
- the notebook is easier to revise before submission,
- the code walkthrough stays aligned with the cleaned `src/` structure.

### Recommended study order
1. Run the data summary and the two visual cells.
2. Read the toy-model section and predict the ranking before running it.
3. Run the subset benchmark and inspect one query in detail.
4. Complete one active-learning prompt without looking at the answer first.
5. Read the PDF report for the full project story and appendix.
